# 04. Sparse, Sliding Window, and Sink Attention (StreamingLLM)

- **Category:** 05_architectures / 04_transformers_and_heads
- **Lineage & Citations:** Iz Beltagy et al., 2020 (*Longformer*); Manzil Zaheer et al., 2020 (*Big Bird*); Guangxuan Xiao et al., 2023 (*StreamingLLM: Efficient Streaming Language Models with Attention Sinks*).
- **Purpose:** Solves the eviction collapse paradox in sliding-window generation by permanently pinning initial "attention sink" tokens, bounding KV cache memory while enabling infinite streaming context.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Flash Attention Online Softmax and Flex Attention](03_flash_attention_online_softmax_and_flex_attention.ipynb) $\to$ `🔗 Downstream Syntheses:` [Transformer Archetypes BERT GPT T5 ViT Swin Perceiver and Jamba](05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> - **Distant In-Context Retrieval:** StreamingLLM permanently discards tokens between sinks and the recent window; cannot recall forgotten middle context without external vector storage.
> - **Window < 64 Tokens:** Overly narrow windows break multi-line code indentation and syntactic hierarchy tracking.

## Mathematical Foundations & Governing Equations

### 1. StreamingLLM Attention Sink Mask
Given query position $i$ and key position $j$, attention is permitted only if $j$ is an initial sink OR falls within the local causal window:
$$
\text{Valid}(i, j) = \left(j < k_{\text{sink}} \lor (i \ge j \land i - j < W_{\text{window}})\right) \land (j \le i)
$$
Mask application and readout:
$$
M_{i, j} = \begin{cases} 0 & \text{if } \text{Valid}(i, j) \\ -\infty & \text{otherwise} \end{cases}, \quad A = \text{softmax}\left(\frac{Q K^\top}{\sqrt{D}} + M\right), \quad O = A V
$$

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/04_transformers_and_heads/04_sparse_sliding_window_and_sink_attention.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

q, k, v = weights["q"], weights["k"], weights["v"]
num_sinks = int(weights["num_sinks"][0])
window_size = int(weights["window_size"][0])
scale = float(weights["scale"][0])
print(f"Loaded {len(weights)} golden tensors on {env.device}, tokens: {list(q.shape)}, sinks: {num_sinks}, window: {window_size}")

In [ ]:
def streaming_sink_attention(q, k, v, num_sinks, window_size, scale):
    B, T, D = q.shape
    S = k.shape[1]
    scores = torch.bmm(q, k.transpose(1, 2)) * scale
    i_idx = torch.arange(T, device=q.device).unsqueeze(1)
    j_idx = torch.arange(S, device=q.device).unsqueeze(0)
    is_sink = j_idx < num_sinks
    is_window = (i_idx - j_idx < window_size) & (i_idx >= j_idx)
    valid_mask = (is_sink | is_window) & (j_idx <= i_idx)
    scores_masked = torch.where(valid_mask.unsqueeze(0), scores, torch.full_like(scores, -1e9))
    attn_weights = F.softmax(scores_masked, dim=-1)
    return torch.bmm(attn_weights, v), attn_weights

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
sink_out, sink_weights = streaming_sink_attention(q, k, v, num_sinks, window_size, scale)
err_o = float(torch.max(torch.abs(sink_out - weights["sink_out"])))
err_w = float(torch.max(torch.abs(sink_weights - weights["sink_weights"])))
assert err_o < 1e-6, f"Sink attention output error: {err_o:.2e}"
assert err_w < 1e-6, f"Sink attention weights error: {err_w:.2e}"

# Verify that initial sink tokens absorb non-zero attention mass for all later queries
sink_mass = sink_weights[:, -1, :num_sinks].sum().item()
assert sink_mass > 0.05, f"Attention sinks must retain significant probability mass: {sink_mass:.4f}"
print(f"Parity Verified: Out={err_o:.2e}, Weights={err_w:.2e}, Final Sink Mass={sink_mass:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_sink = benchmark_fn(lambda: streaming_sink_attention(q, k, v, num_sinks, window_size, scale), iters=50, name="Streaming Sink Attn")
print(format_benchmark_table([b_sink]))

# Visualization: StreamingLLM Attention Sparsity Pattern
fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
ax[0].imshow(sink_weights[0].detach().numpy(), cmap="viridis", aspect="auto")
ax[0].set_title("Streaming Attention Matrix (Sinks + Window)")
ax[1].plot(sink_weights[0, :, 0].detach().numpy(), marker="o", color="teal")
ax[1].set_title("Attention Sink Token 0 Mass across Queries")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse: Naive Window Eviction (Zero Sinks)
# Without attention sinks, the model is forced to over-concentrate attention on arbitrary boundary tokens!
_, naive_weights = streaming_sink_attention(q, k, v, num_sinks=0, window_size=window_size, scale=scale)
print(f"Max token weight without sinks: {naive_weights[0, -1].max().item():.3f}")
print(f"Max token weight with sinks: {sink_weights[0, -1].max().item():.3f}")
# The distribution shifts drastically when sinks are removed
dist_shift = float(torch.norm(sink_weights - naive_weights))
assert dist_shift > 0.2, "Removing sinks must produce substantial distributional distortion!"
print("Demonstrated requirement of attention sinks to preserve stable attention dynamics.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Initial Tokens are Sinks:** Softmax requires initial tokens to absorb residual probability mass.
2. **StreamingLLM Enables Infinite Horizon:** Retaining 4 sink tokens and a local window guarantees stable generation indefinitely.
3. **Zero Fine-Tuning Required:** Works out-of-the-box on pretrained Llama, Mistral, and Falcon models.

---
### Cross-Domain Generalisation Matrix
- **Continuous 24/7 Agent Operation:** Chatbots stream tokens for days without restart or memory exhaustion.
- **Real-Time Speech Processing:** Streaming audio codecs retain initial voice calibration sinks.
- **Edge Embedded NPUs:** Fits unlimited conversational turns into fixed 4GB RAM allocations.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_sparse_sliding_window_and_sink_attention.md](../../../markdown/05_architectures/04_transformers_and_heads/04_sparse_sliding_window_and_sink_attention.md)
- **PyTorch 2.x Module:** [04_sparse_sliding_window_and_sink_attention.py](../../../pytorch2/05_architectures/04_transformers_and_heads/04_sparse_sliding_window_and_sink_attention.py)
- **Functional JAX Module:** [04_sparse_sliding_window_and_sink_attention.py](../../../jax/05_architectures/04_transformers_and_heads/04_sparse_sliding_window_and_sink_attention.py)
- **Candle Rust Module:** [04_sparse_sliding_window_and_sink_attention.rs](../../../candle/05_architectures/04_transformers_and_heads/04_sparse_sliding_window_and_sink_attention.rs)
- **Hardware Kernel:** [04_sparse_sliding_window_and_sink_attention.rs](../../../kernels/05_architectures/04_transformers_and_heads/04_sparse_sliding_window_and_sink_attention.rs)
- **Universal Golden SafeTensors:** [04_sparse_sliding_window_and_sink_attention.safetensors](../../../weights/05_architectures/04_transformers_and_heads/04_sparse_sliding_window_and_sink_attention.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 03. Flash Attention Online Softmax and Flex Attention](03_flash_attention_online_softmax_and_flex_attention.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 05. Transformer Archetypes BERT GPT T5 ViT Swin Perceiver and Jamba](05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.ipynb) &rarr;